Version 2: Circuit that constructs the superposition: $\sum_{2^I}(|Window_i>|Index_i>|Grid>)$
- N-dimensional network. -> (New Improvement)
- Initialization of possible valid indices. -> (New Improvement)
- Gray mask on idx (We cancel gates: XX -> I).

In [ ]:
import time
import qiskit
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, SparsePauliOp, partial_trace
from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere
from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.visualization import plot_histogram
from itertools import product
from math import prod

print(qiskit.__version__)

In [ ]:
# =========================================================
# FUNCIONES AUXILIARES GENERALES PARA D DIMENSIONES
# =========================================================

def index_to_coord(index, dims):
    """
    Convierte un indice lineal row-major a coordenada D-dimensional.
    """
    coord = [0] * len(dims)
    rem = index
    for d in reversed(range(len(dims))):
        coord[d] = rem % dims[d]
        rem //= dims[d]
    return tuple(coord)

def coord_to_index(coord, dims):
    """
    Convierte una coordenada D-dimensional a indice lineal row-major.
    """
    idx_lin = 0
    stride = 1
    for d in reversed(range(len(dims))):
        idx_lin += coord[d] * stride
        stride *= dims[d]
    return idx_lin

def reshape_bits_nd(bitstring, dims):
    """
    Reorganiza un bitstring lineal row-major a una estructura anidada
    con forma dims.
    """
    arr = np.array(list(bitstring), dtype=str)
    return arr.reshape(dims)

def format_nd_array(arr, indent=0):
    """
    Formatea un ndarray/string-array de cualquier dimension de forma legible.
    """
    if arr.ndim == 1:
        return "[" + " ".join(arr.tolist()) + "]"

    inner = []
    for sub in arr:
        inner.append(format_nd_array(np.array(sub), indent + 2))

    sep = ",\n" + " " * indent
    return "[\n" + " " * indent + sep.join(inner) + "\n" + " " * max(indent - 2, 0) + "]"

def valid_starts_nd(N, M):
    """
    Lista de coordenadas iniciales validas para la ventana en D dimensiones.
    Usa la misma convencion row-major.
    """
    shape_starts = tuple(N[d] - M[d] + 1 for d in range(len(N)))
    return list(np.ndindex(shape_starts))

def window_qubits_nd(start, N, M):
    """
    Devuelve los indices lineales de los qubits de la ventana que empieza en 'start',
    en orden row-major dentro del bloque M.

    Ejemplo 2D con M=[2,2]:
    offsets visitados:
        (0,0), (0,1), (1,0), (1,1)
    """
    qubits = []
    for offset in np.ndindex(tuple(M)):
        coord = tuple(start[d] + offset[d] for d in range(len(N)))
        qubits.append(coord_to_index(coord, N))
    return qubits

In [ ]:
# =========================
# Parametros
# =========================

D = 2
N = [4, 4]   # tamaño de la red en cada dimension
M = [2, 2]   # tamaño del trabajo en cada dimension

if len(N) != D or len(M) != D:
    raise ValueError("N y M deben tener longitud D")

for d in range(D):
    if M[d] > N[d]:
        raise ValueError(f"En la dimension {d}, M[{d}] no puede ser mayor que N[{d}]")

# Numero total de celdas (red y trabajo) y numero de ventanas validas en D dimensiones
N_tot = prod(N)
M_tot = prod(M)
starts = valid_starts_nd(N, M)
W = len(starts)

# Qubits indice necesarios
k = int(np.ceil(np.log2(W))) if W > 1 else 1

print(f"Dimensiones: {D}")
print(f"Tamano de la red por dimension: {N}")
print(f"Tamano del trabajo por dimension: {M}")
print(f"Total de celdas: {N_tot}")
print(f"Total de celdas del trabajo: {M_tot}")
print(f"Total de ventanas posibles: {W}")
print(f"Qubits necesarios para representar las ventanas: {k}")

print("\nVentanas validas posibles (coordenada inicial):")
for idx_start, start in enumerate(starts):
    print(f"{idx_start}: {start} -> {window_qubits_nd(start, N, M)}")

# =========================
# Registros
# =========================

n = QuantumRegister(N_tot, "n")   # red
idx = QuantumRegister(k, "i")     # indice de ventana
m = QuantumRegister(M_tot, "m")   # ventana de trabajo

qc = QuantumCircuit(n, idx, m)

# =========================
# Estado inicial de ejemplo
# =========================

# Cuadrado 2x2 en (0,0) en N=4x4
# qc.x(n[0])
# qc.x(n[1])
# qc.x(n[4])
# qc.x(n[5])

# Cuadrado 2x2 en (2,1) en N=4x4
# qc.x(n[9])
# qc.x(n[10])
# qc.x(n[13])
# qc.x(n[14])

# # Esquinas en N=3x3
# qc.x(n[0])
# qc.x(n[2])
# qc.x(n[6])
# qc.x(n[8])

# Esquinas en N=4x4
qc.x(n[0])
qc.x(n[3])
qc.x(n[12])
qc.x(n[15])

# =========================
# Superposicion de indices SOLO validos
# =========================
# Prepara:
#   (1/sqrt(W)) * sum_{i=0}^{W-1} |i>
# y deja amplitud 0 en los indices invalidos i >= W

amps_idx = np.zeros(2**k, dtype=complex)
amps_idx[:W] = 1 / np.sqrt(W)
qc.initialize(amps_idx, idx)

# =========================
# Orden Gray-like para minimizar X en idx
# =========================
# Recorremos los indices validos en un orden cercano a Gray:
# tomamos la secuencia Gray completa sobre k bits y filtramos los validos < W

gray_full = [t ^ (t >> 1) for t in range(2**k)]
order_valid = [g for g in gray_full if g < W]

print("\nOrden Gray-like usado para cargar ventanas:")
print(order_valid)

# =========================
# Carga coherente de ventanas
# =========================
# En vez de:
#   activar X segun i
#   hacer MCX
#   deshacer X
# para cada i,
# mantenemos el patron actual de X sobre idx y solo cambiamos
# los bits que realmente varian entre una ventana y la siguiente.

current_zero_mask = [False] * k  # False = no hay X puesta en idx[b]

for i in order_valid:
    bits = [(i >> b) & 1 for b in range(k)]   # little-endian
    target_zero_mask = [bits[b] == 0 for b in range(k)]
    win = window_qubits_nd(starts[i], N, M)

    # Solo togglear los idx[b] cuyo estado X/no-X cambie
    for b in range(k):
        if current_zero_mask[b] != target_zero_mask[b]:
            qc.x(idx[b])
            current_zero_mask[b] = target_zero_mask[b]

    # Copiar XOR de la ventana sobre m
    for j, n_pos in enumerate(win):
        controls = [idx[b] for b in range(k)] + [n[n_pos]]
        qc.mcx(controls, m[j])

# Limpiar el patron final de X sobre idx
for b in range(k):
    if current_zero_mask[b]:
        qc.x(idx[b])

qc.draw(output='mpl')

In [ ]:
sv = Statevector(qc)
sv.draw(output='latex', max_size=2**qc.num_qubits, prefix="|\\psi\\rangle = ")

In [ ]:
# =========================================================
# 1) Quitar el registro n
# =========================================================
rho = partial_trace(sv, list(range(N_tot)))

# =========================================================
# 2) Comprobar pureza
# =========================================================
purity = np.real(np.trace(rho.data @ rho.data))
print(f"Purity = {purity}")

if not np.isclose(purity, 1.0, atol=1e-10):
    raise ValueError("El estado reducido no es puro; no se puede representar como un unico ket.")

# =========================================================
# 3) Obtener el array completo de posiciones n
# =========================================================
tol_n = 1e-10
total_q = qc.num_qubits
array_posiciones = None

for basis_idx, amp in enumerate(sv.data):
    if abs(amp) <= tol_n:
        continue

    bits_full = format(basis_idx, f'0{total_q}b')

    # En Qiskit, el string sale como:
    # m_{M_tot-1}...m_0  i_{k-1}...i_0  n_{N_tot-1}...n_0
    # Por eso los qubits n estan al final.
    n_desc = bits_full[-N_tot:]    # n_{N_tot-1}...n_0
    n_asc = n_desc[::-1]           # n_0...n_{N_tot-1}

    if array_posiciones is None:
        array_posiciones = n_asc
    elif n_asc != array_posiciones:
        raise ValueError("Los qubits n no son deterministas (no hay un unico array de posiciones).")

if array_posiciones is None:
    raise ValueError("No se pudo inferir el array de posiciones desde el statevector.")

print(f"Array de posiciones lineal = {array_posiciones}\n")

# Mostrar tambien la red con forma N
red_nd = reshape_bits_nd(array_posiciones, N)
print(f"Red con forma {tuple(N)}:")
print(format_nd_array(red_nd))
print()

# =========================================================
# 4) Extraer el ket del estado reducido
# =========================================================
vals, vecs = np.linalg.eigh(rho.data)
psi_red = Statevector(vecs[:, np.argmax(vals)])

# =========================================================
# 5) Extraer etiquetas del estado reducido
# =========================================================
# En psi_red el string binario tiene la forma:
# m_{M_tot-1}...m_0  i_{k-1}...i_0
total_bits = M_tot + k
tol = 1e-10
filas = []

for basis_idx, amp in enumerate(psi_red.data):
    if abs(amp) <= tol:
        continue

    bits = format(basis_idx, f'0{total_bits}b')

    m_desc = bits[:M_tot]     # m_{M_tot-1}...m_0
    i_desc = bits[M_tot:]     # i_{k-1}...i_0

    ventana_lineal = m_desc[::-1]   # m_0...m_{M_tot-1}
    indices = i_desc
    idx_int = int(indices, 2)

    filas.append((idx_int, indices, ventana_lineal))

# Ordenar por valor entero del indice
filas.sort(key=lambda x: (x[0], x[2]))

# Separar indices validos e invalidos
filas_validas = [fila for fila in filas if fila[0] < W]
filas_invalidas = [fila for fila in filas if fila[0] >= W]

# =========================================================
# 6) Mostrar ventanas validas con forma ND
# =========================================================
print("Ventanas validas:\n")

for idx_int, indices, ventana_lineal in filas_validas:
    start = starts[idx_int]   # coordenada inicial ND correspondiente a ese indice
    ventana_nd = reshape_bits_nd(ventana_lineal, M)

    print(f"Indice: {indices} ({idx_int})")
    print(f"Start:  {start}")
    print(f"Ventana lineal: {ventana_lineal}")
    print(f"Ventana con forma {tuple(M)}:")
    print(format_nd_array(ventana_nd))
    print()

# =========================================================
# 7) Mostrar indices invalidos, si existen
# =========================================================
if filas_invalidas:
    print("----------------------------------------")
    print("Indices fuera de ventana valida:\n")

    for idx_int, indices, ventana_lineal in filas_invalidas:
        ventana_nd = reshape_bits_nd(ventana_lineal, M)

        print(f"Indice: {indices} ({idx_int})")
        print(f"Ventana lineal: {ventana_lineal}")
        print(f"Ventana con forma {tuple(M)}:")
        print(format_nd_array(ventana_nd))
        print()